In [ ]:
#@markdown Setup dependencies (Colab only)
%%capture
try:
    import papyrus_scripts
except ImportError:
    !pip uninstall papyrus-scripts -y
    !pip install rdkit
    !pip install --upgrade papyrus-scripts --no-cache-dir
    get_ipython().kernel.do_shutdown(True)

# 🚀 Advanced examples: using Papyrus-scripts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OlivierBeq/Papyrus-scripts/blob/master/notebook_examples/advanced_querying.ipynb)

This notebook covers two more advanced topics, each demonstrated through both APIs:

1. [🔍 Similarity & substructure search](#1) over the Papyrus compound structures — chemical-space filtering beyond simple column matching.
2. [🧠 QSAR, PCM & DNN modelling](#2) of the bioactivity data — building predictive models directly on curated subsets.

> ⚠️ Both topics need extra data or dependencies: similarity/substructure search needs the compound structures downloaded and the `papyrus-scripts[subsim]` (or `[gpu]`) extra; modelling needs precomputed descriptors and, for XGBoost/DNN respectively, `xgboost`/`papyrus-scripts[dnn]`.

<a id="1"></a>
## 1. 🔍 Similarity & substructure search

Behind the scenes, both APIs below build an indexed [FPSim2](https://github.com/UCLCheminformatics/FPSim2)/RDKit search database (an `FPSubSim2` `.h5` file) from the Papyrus structures:
- **similarity search** ranks compounds by Tanimoto coefficient against a query fingerprint;
- **substructure search** finds compounds containing a query substructure exactly (subgraph isomorphism), not just similar ones.

Building the database requires the structures file to have been downloaded first — `.molecules()` (used below) does that for us.

In [ ]:
from papyrus_scripts import PapyrusDataset

dataset = PapyrusDataset(version='latest', plusplus=True, is3d=False)

# Downloads the compound structures if not already present locally
_ = dataset.molecules().to_dataframe()

### 🧩 Object-oriented API

`keep_similar_molecules`, `keep_dissimilar_molecules`, `keep_substructure_molecules` and `keep_not_substructure_molecules` transparently build (once) and reuse the underlying `FPSubSim2` search database — no manual setup required.

| Parameter | Meaning |
|---|---|
| `smiles` | one or more query SMILES strings |
| `fp` | fingerprint type for similarity search (default: Morgan/ECFP-like) |
| `threshold` | minimum Tanimoto coefficient to keep (similarity search only) |
| `cuda` | use a GPU-accelerated search engine (default `False`) |

In [ ]:
caffeine = 'Cn1cnc2c1c(=O)n(C)c(=O)n2C'

similar = (dataset
           .keep_similar_molecules(caffeine, threshold=0.6, cuda=False)
           .to_dataframe())
print(f'🔎 Similar to caffeine (Tanimoto ≥ 0.6): {similar.shape[0]} activity points')

substructures = (dataset
                 .keep_substructure_molecules('c1ccccc1')  # any molecule containing a benzene ring
                 .to_dataframe())
print(f'🔎 Containing a benzene ring: {substructures.shape[0]} activity points')

> 🖥️➡️🎮 Pass `cuda=True` to use a GPU-accelerated search engine instead (requires `papyrus-scripts[gpu]` and a CUDA-capable GPU).

### ⚙️ Pre-building the search database explicitly

The filters above call `create_fp_subsim_search()` internally, with default arguments, the first time they run against a dataset whose `.h5` database doesn't exist yet. Call it directly beforehand to control how that database is built:

| Parameter | Meaning |
|---|---|
| `fp` | fingerprint(s) to store (default: Morgan/ECFP-like) |
| `path` | explicit `.h5` output path (default: auto-derived) |
| `progress` | show progress bars while building |
| `njobs` | worker processes for fingerprint computation (`-1` = all cores) |
| `n_shards` | substructure-library shard count (single-process builds only) |
| `pattern_holder_bits` | prescreen size for the substructure library (size vs. query-speed tradeoff) |
| `force` | rebuild even if the database file already exists |

In [ ]:
from papyrus_scripts.fingerprint import MorganFingerprint

# Pre-build the database with a custom fingerprint, using all CPU cores
db_path = dataset.create_fp_subsim_search(fp=MorganFingerprint(), njobs=-1)
print(f'📦 FPSubSim2 database ready at: {db_path}')

### 📚 Function API

The lower-level functions operate on a plain DataFrame/LazyFrame and require an explicit path to an `FPSubSim2` `.h5` database — useful for reusing one database across many independent filter calls, or for combining with the rest of the functional filtering API.

In [ ]:
from papyrus_scripts.subsim_search import FPSubSim2
from papyrus_scripts import read_papyrus, keep_quality, keep_similar, keep_substructure, consume_chunks

# Build (or reuse) the search database explicitly
fpss = FPSubSim2()
fpss.create_from_papyrus(version='latest', is3d=False, njobs=-1)  # all CPU cores

sample_data = read_papyrus(is3d=False, plusplus=True, chunksize=1_000_000, source_path=None)
filtered = consume_chunks(keep_quality(sample_data, min_quality='medium'), progress=True)

similar = keep_similar(filtered, molecule_smiles=caffeine, fpsubsim2_file=fpss.h5_filename, threshold=0.6)
substructures = keep_substructure(filtered, molecule_smiles='c1ccccc1', fpsubsim2_file=fpss.h5_filename)

`FPSubSim2` also gives direct access to the search engines for full control — CPU/GPU/auto-fallback, RAM-light on-disk search, and exact substructure matches with their Papyrus identifiers. To reuse an existing database in a later session, load it first with `fpss.load('/path/to/database.h5')`.

In [ ]:
# cuda=False (default, CPU) | True (GPU, raises if unavailable) | 'auto' (GPU with CPU fallback)
engine = fpss.get_similarity_lib(cuda='auto')
hits = engine.similarity(caffeine, threshold=0.6)
hits.head()

In [ ]:
sub_lib = fpss.get_substructure_lib()
matches = sub_lib.substructure('c1ccccc1')
matches.head()

<a id="2"></a>
## 2. 🧠 Modelling: QSAR, PCM and DNN

All modelling functions live under `papyrus_scripts.modelling` and operate on a plain DataFrame (typically produced by either API above) — there is currently no `PapyrusDataset` wrapper for them.

| | QSAR | PCM |
|---|---|---|
| Features | molecular descriptors only | molecular **and** protein descriptors |
| One model per | target | *all* targets in the data at once |
| Best for | a single well-studied target | generalizing (and extrapolating) across related targets |

Let's prepare a small, fast-to-model dataset: activity of the human and rat serotonin transporter.

In [ ]:
from papyrus_scripts import PapyrusDataset

model_dataset = (PapyrusDataset(version='latest', plusplus=True)
                 .keep_accession(['P31645', 'P31652'])  # human & rat SLC6A4
                 .keep_quality('medium')
                 .keep_activity_type(['Ki', 'KD']))

model_data = model_dataset.to_dataframe()
model_data.shape

### 📈 QSAR

QSAR (Quantitative Structure-Activity Relationship) models predict bioactivity from molecular descriptors alone. `qsar()` handles descriptor loading, train/test splitting, cross-validation and evaluation in one call; it defaults to an `xgboost.XGBRegressor`/`XGBClassifier` but accepts any scikit-learn-compatible estimator via `model=`.

A handful of parameters worth understanding before tuning them:

| Parameter | Meaning |
|---|---|
| `num_points` | minimum activity points required for a target to be modelled at all |
| `delta_activity` | minimum spread between the most and least active compound for a target to be modelled |
| `activity_threshold` | active/inactive cutoff used to binarize `endpoint` (classifiers only; ignored by regressors) |
| `split_by` | how the test set is carved out: `'random'`, `'Year'` (temporal), `'cluster'`, `'custom-cluster'`, or `'custom'` |
| `folds` | number of cross-validation folds on the remaining training data |

In [ ]:
import xgboost
from papyrus_scripts.modelling import qsar

reg_results, reg_models = qsar(data=model_data,
                                endpoint='pchembl_value_Mean',
                                num_points=30,
                                delta_activity=2,
                                descriptors='mold2',
                                activity_threshold=6.5,
                                model=xgboost.XGBRegressor(verbosity=0),
                                folds=5,
                                split_by='Year',
                                split_year=2013,
                                random_state=1234,
                                verbose=True)
reg_results

Training a classifier instead is simply a matter of passing a classifier model — `pchembl_value_Mean` is binarized around `activity_threshold` for evaluation.

In [ ]:
cls_results, cls_models = qsar(data=model_data,
                                endpoint='pchembl_value_Mean',
                                descriptors='mold2',
                                activity_threshold=6.5,
                                model=xgboost.XGBClassifier(verbosity=0),
                                folds=5,
                                split_by='Year',
                                split_year=2013,
                                random_state=1234,
                                verbose=True)
cls_results

### 🧬 PCM

Proteochemometric (PCM) models additionally encode the protein target as descriptors, letting a **single** model generalize across every target present in the data — here, the human and rat transporter jointly, instead of two separate QSAR models.

In [ ]:
from papyrus_scripts.modelling import pcm

pcm_results, pcm_models = pcm(data=model_data,
                               endpoint='pchembl_value_Mean',
                               mol_descriptors='mold2',
                               prot_descriptors='unirep',
                               activity_threshold=6.5,
                               model=xgboost.XGBRegressor(verbosity=0),
                               folds=5,
                               split_by='Year',
                               split_year=2013,
                               random_state=1234,
                               verbose=True)
pcm_results

### 🎲 y-scrambling

Passing `yscramble=True` to either `qsar()` or `pcm()` randomly permutes the target variable before training. It's a standard sanity check: a model that performs well on real labels should perform close to randomly on scrambled ones — if it doesn't, that's a red flag for data leakage.

In [ ]:
scrambled_results, _ = qsar(data=model_data,
                            endpoint='pchembl_value_Mean',
                            descriptors='mold2',
                            activity_threshold=6.5,
                            model=xgboost.XGBRegressor(verbosity=0),
                            yscramble=True,
                            random_state=1234,
                            verbose=False)
scrambled_results

### 🔁 Repeating training over multiple seeds

`qsar()`/`pcm()` do not repeat training internally — `random_state` controls a single train/test split and cross-validation shuffle. To assess how sensitive results are to that choice, call them in a loop over several seeds and inspect the spread:

In [ ]:
import pandas as pd

all_results = []
for seed in range(5):
    perf, _ = qsar(data=model_data,
                   endpoint='pchembl_value_Mean',
                   descriptors='mold2',
                   activity_threshold=6.5,
                   model=xgboost.XGBRegressor(verbosity=0),
                   random_state=seed,
                   verbose=False)
    all_results.append(perf.reset_index().assign(seed=seed))

all_results = pd.concat(all_results, ignore_index=True)
all_results.head()

### 🤖 Deep neural networks (DNN)

`papyrus_scripts.neuralnet` provides single- and multi-task PyTorch/skorch estimators (requires the `papyrus-scripts[dnn]` extra):

| Class | Use case |
|---|---|
| `SingleTaskNNRegressor` / `SingleTaskNNClassifier` | one target, continuous / binary endpoint |
| `MultiTaskNNRegressor` / `MultiTaskNNClassifier` | several targets predicted jointly (`n_task >= 2`) |

> ⚠️ Unlike scikit-learn/XGBoost models, these need `set_architecture()` (and, for early stopping, `set_validation()`) called explicitly before `.fit()`. `qsar()`/`pcm()`'s internal cross-validation loop does **not** call these for you, so DNN models are used standalone below rather than passed as their `model=` argument.

In [ ]:
from sklearn.model_selection import train_test_split
from papyrus_scripts.neuralnet import SingleTaskNNRegressor

# Reuse the object-oriented API to fetch descriptors for the same filtered compounds
descriptors = model_dataset.molecular_descriptors('mold2').to_dataframe()
data = model_data.join(descriptors, on='connectivity')

feature_cols = [c for c in descriptors.columns if c != 'connectivity']
X = data.select(feature_cols).to_numpy()
y = data['pchembl_value_Mean'].to_numpy()

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=1234)

reg = SingleTaskNNRegressor(out='./dnn_checkpoints', epochs=50, lr=1e-3, hidden_layers=[512, 128])
reg.set_architecture(n_dim=X_train.shape[1])
reg.set_validation(X_valid, y_valid)
reg.fit(X_train, y_train)

predictions = reg.predict(X_valid)

For multi-target models, use `MultiTaskNNRegressor`/`MultiTaskNNClassifier` with `set_architecture(n_dim, n_task)` and a multi-column target array instead.

---

🎉 That's the full tour! For everyday filtering, see [simple_examples.ipynb](https://github.com/OlivierBeq/Papyrus-scripts/blob/master/notebook_examples/simple_examples.ipynb); for matching against the PDB, see [matchRCSB.ipynb](https://github.com/OlivierBeq/Papyrus-scripts/blob/master/notebook_examples/matchRCSB.ipynb).